# Research 72 — RealMLP-TD full-refit Test predictions (Vast.ai GPU)

**Purpose:** produce Test predictions for the RealMLP-TD candidate. Notebook 71 did not validate it under the rule set in advance: RealMLP alone beat the submitted recipe overall on all three new splits, but one fold slice (split 31415, fold 2) was −0.0003. These predictions are **not a submission**.

**Recipe:** the notebook 65 setup (RealMLP-TD defaults, `random_state=42`, the tuned feature set). It is fitted once on all 104,000 labeled rows. Numeric medians come from those same rows. Test rows are then predicted. The same settings were used per fold in notebooks 65 and 67–70.

**Outputs:** `predictions/segment-realmlp-td-test.csv`, keyed by `player_id`, with `prob_0`–`prob_3` and `segment` (argmax). The file is row-aligned to `test.csv`. `submit/index.csv` is not touched. Map `player_id` to `ANS` by test row position when building a submission.

Runs on the Vast.ai RTX 3090 (torch 2.14.1 cu130, pytabkit 1.8.0, pandas 3.1.0, numpy 2.5.4, scikit-learn 1.9.1).

In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from pytabkit import RealMLP_TD_Classifier

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
train = pd.read_csv(ROOT / "data/train.csv")
test = pd.read_csv(ROOT / "data/test.csv")
y = train.segment.to_numpy(dtype=int)
assert "segment" not in test.columns and test["player_id"].is_unique
assert set(np.unique(y)) == {0, 1, 2, 3}
print("train", train.shape, "| test", test.shape)

/workspace/ml00x-task2/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


train (104000, 46) | test (25889, 45)


In [2]:
def features(frame):
    x = frame.drop(
        columns=["segment", "player_id", "random_metric_1", "random_metric_2", "random_metric_3"],
        errors="ignore",
    ).copy()
    ratios = {
        "spend_per_hour": ("total_spending_thb", "total_playtime_hours"),
        "spend_per_account_day": ("total_spending_thb", "account_age_days"),
        "playtime_per_account_day": ("total_playtime_hours", "account_age_days"),
        "monthly_spend_per_purchase": ("avg_monthly_spending", "spending_frequency"),
        "gifts_per_friend": ("gifts_sent_received", "friend_count"),
        "invites_per_friend": ("friend_invites_sent", "friend_count"),
        "ranked_per_team": ("ranked_participation_rate", "team_play_percentage"),
    }
    for name, (a, b) in ratios.items():
        x[name] = x[a] / (x[b].abs() + 1)
    x["weekly_minutes"] = x["play_frequency"] * x["avg_session_duration"]
    x["social_activity"] = x["friend_count"] * x["team_play_percentage"]
    x["missing_count"] = x.isna().sum(axis=1)
    return x.replace([np.inf, -np.inf], np.nan)


X = features(train)
X_test = features(test)
assert list(X.columns) == list(X_test.columns)
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]
print("features", X.shape[1], "categorical", len(cat_cols), "numeric", len(num_cols))

features 51 categorical 10 numeric 41


In [3]:
started = time.perf_counter()
x_fit = X.copy()
x_pred = X_test.copy()
x_fit[cat_cols] = x_fit[cat_cols].fillna("__MISSING__").astype(str)
x_pred[cat_cols] = x_pred[cat_cols].fillna("__MISSING__").astype(str)
medians = x_fit[num_cols].median()
x_fit[num_cols] = x_fit[num_cols].fillna(medians)
x_pred[num_cols] = x_pred[num_cols].fillna(medians)
model = RealMLP_TD_Classifier(device="cuda", random_state=42, n_threads=4, verbosity=0)
model.fit(x_fit, y, cat_col_names=cat_cols)
p_test = model.predict_proba(x_pred)
elapsed = time.perf_counter() - started
print(
    f"Fit and predict elapsed seconds={elapsed:.1f}. Fit on all labeled rows; test.csv used for prediction only."
)

/workspace/ml00x-task2/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Fit and predict elapsed seconds=698.9. Fit on all labeled rows; test.csv used for prediction only.


/workspace/ml00x-task2/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


In [4]:
assert p_test.shape == (len(test), 4) and np.isfinite(p_test).all()
assert np.allclose(p_test.sum(axis=1), 1.0, atol=1e-5)
predictions = pd.DataFrame(
    {
        "player_id": test["player_id"].astype(str),
        **{f"prob_{i}": p_test[:, i] for i in range(4)},
        "segment": p_test.argmax(axis=1).astype(int),
    }
)
assert predictions["player_id"].equals(test["player_id"].astype(str))
predictions.to_csv(ROOT / "predictions" / "segment-realmlp-td-test.csv", index=False)

train_share = pd.Series(y).value_counts(normalize=True).sort_index().round(4).to_dict()
test_share = predictions["segment"].value_counts(normalize=True).sort_index().round(4).to_dict()
summary = {
    "test_rows": int(len(test)),
    "fit_rows": int(len(train)),
    "elapsed_seconds": round(elapsed, 1),
    "train_class_share": {str(k): float(v) for k, v in train_share.items()},
    "test_predicted_class_share": {str(k): float(v) for k, v in test_share.items()},
    "test_loaded_for": "prediction only",
    "submission_status": "not written to submit/index.csv",
}
(ROOT / "assets" / "segment-realmlp-td-test.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)
print(json.dumps(summary, indent=2))

{
  "test_rows": 25889,
  "fit_rows": 104000,
  "elapsed_seconds": 698.9,
  "train_class_share": {
    "0": 0.3936,
    "1": 0.2503,
    "2": 0.2021,
    "3": 0.154
  },
  "test_predicted_class_share": {
    "0": 0.4345,
    "1": 0.2409,
    "2": 0.1825,
    "3": 0.1422
  },
  "test_loaded_for": "prediction only",
  "submission_status": "not written to submit/index.csv"
}


## Status

These predictions are a candidate only. Whether to submit them is a separate decision. RealMLP was not validated under the rule set in notebook 71.